# Mask R-CNN Preprocessing - Rice/Coffee Leaf Disease

Standalone Kaggle notebook for the raw dataset `magnusdtd2/rice-coffee-leaf-disease`.

Output layout:

- `/kaggle/working/data/mask_rcnn_processed/images/{train,val,test}/...`
- `/kaggle/working/data/mask_rcnn_processed/annotations/{train,val,test}.coco.json`
- `/kaggle/working/data/mask_rcnn_processed/metadata/*.json|*.csv`
- Optional zip: `/kaggle/working/mask_rcnn_processed.zip` when `CREATE_ZIP=True`

Runtime notes:

- Default disables blur filtering because the raw images are large and full-image blur scoring is expensive.
- Processing still resizes, pads, transforms COCO bbox/polygon geometry, drops missing/zero annotations, and computes train normalization.
- Expected runtime on Kaggle CPU is usually minutes, not seconds, because thousands of images are decoded/resized and zipped.

The zip step is disabled by default. Kaggle can keep `/kaggle/working/data/mask_rcnn_processed` as notebook output, and the training notebooks can read that directory directly.


In [1]:
from pathlib import Path

SEED = 42
IMAGE_SIZE = 384
DROP_BLURRY = False  # Fast Kaggle default. Set True only for a slower quality audit.
BLUR_QUANTILE = 0.02
OVERWRITE = True

# The Kaggle mount can be either root-level or nested under data/.
RAW_ROOT_CANDIDATES = [
    Path("/kaggle/input/rice-coffee-leaf-disease"),
    Path("/kaggle/input/rice-coffee-leaf-disease/data"),
    Path("/kaggle/input/rice-coffee-leaf-disease/Data"),
    Path("/kaggle/input/rice-coffee-leaf-disease/raw"),
    Path("/kaggle/input"),
    Path("archive (2)"),
    Path("archive (2)/data"),
]

OUTPUT_ROOT = Path("/kaggle/working/data/mask_rcnn_processed")
ZIP_OUTPUT = Path("/kaggle/working/mask_rcnn_processed")
CREATE_ZIP = False  # Set True only if you need a downloadable zip; zipping can run for a long time on Kaggle.


In [2]:
"""Prepare a Mask R-CNN-ready COCO dataset for rice/coffee leaf disease.

The general preprocessing notebook is classification-oriented in a few places.
This script keeps image, bbox, and polygon geometry aligned for instance
segmentation and writes a clean processed dataset for the Mask R-CNN notebooks.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

from PIL import Image, ImageFile, ImageOps, UnidentifiedImageError

import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

ImageFile.LOAD_TRUNCATED_IMAGES = True

RICE_CLASSES = ["Healthy", "BrownSpot", "Hispa", "LeafBlast"]
COFFEE_CLASSES = ["LeafMiner", "PowderyMildew", "Rust", "AlgalLeafSpot"]
CLASS_NAMES = RICE_CLASSES + COFFEE_CLASSES
CLASS_TO_CATEGORY_ID = {name: idx + 1 for idx, name in enumerate(CLASS_NAMES)}
COCO_CATEGORIES = [{"id": idx + 1, "name": name} for idx, name in enumerate(CLASS_NAMES)]
VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


@dataclass(frozen=True)
class PreprocessConfig:
    raw_root: Path
    output_root: Path
    image_size: int = 384
    seed: int = 42
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    drop_blurry: bool = True
    blur_quantile: float = 0.02
    jpeg_quality: int = 95
    overwrite: bool = False


def normalize_coco_name(name: str) -> str:
    return str(name).replace("\\", "/").lstrip("./")


def label_dirname(label: str) -> str:
    return label.strip().replace(" ", "_").replace("/", "-")


def md5_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_image_info(path: Path) -> dict[str, Any]:
    try:
        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode
        return {"width": width, "height": height, "mode": mode, "is_valid": True}
    except (UnidentifiedImageError, OSError, ValueError):
        return {"width": "", "height": "", "mode": "invalid", "is_valid": False}


def blur_score(path: Path) -> float:
    """Fast blur proxy on a downsampled grayscale image."""
    try:
        with Image.open(path) as img:
            gray = ImageOps.grayscale(img)
            gray.thumbnail((256, 256), Image.Resampling.BILINEAR)
            arr = np.asarray(gray, dtype=np.float32)
        if arr.shape[0] < 3 or arr.shape[1] < 3:
            return 0.0
        lap = (
            -4.0 * arr[1:-1, 1:-1]
            + arr[1:-1, :-2]
            + arr[1:-1, 2:]
            + arr[:-2, 1:-1]
            + arr[2:, 1:-1]
        )
        return float(lap.var())
    except Exception:
        return float("nan")


def percentile(values: list[float], q: float) -> float:
    clean = sorted(v for v in values if not math.isnan(v))
    if not clean:
        return 0.0
    pos = (len(clean) - 1) * q
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return clean[lo]
    return clean[lo] * (hi - pos) + clean[hi] * (pos - lo)


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def build_source_annotation_index(domain: str, annotation_path: Path) -> dict[tuple[str, str], list[dict[str, Any]]]:
    coco = load_json(annotation_path)
    images_by_id = {int(img["id"]): img for img in coco.get("images", [])}
    anns_by_file: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
    for ann in coco.get("annotations", []):
        img = images_by_id.get(int(ann.get("image_id", -1)))
        if not img:
            continue
        file_name = normalize_coco_name(img.get("file_name", ""))
        anns_by_file[(domain, file_name)].append(ann)
    return dict(anns_by_file)


def collect_records(raw_root: Path) -> list[dict[str, Any]]:
    domain_specs = [
        ("rice", raw_root / "rice_leaf_disease", {label: label for label in RICE_CLASSES}),
        ("coffee", raw_root / "coffee_leaf_disease", {str(idx): label for idx, label in enumerate(COFFEE_CLASSES)}),
    ]
    records: list[dict[str, Any]] = []
    sample_index = 0
    for domain, domain_dir, folder_to_label in domain_specs:
        if not domain_dir.exists():
            raise FileNotFoundError(f"Missing domain directory: {domain_dir}")
        for folder_name, label in folder_to_label.items():
            class_dir = domain_dir / folder_name
            if not class_dir.exists():
                raise FileNotFoundError(f"Missing class directory: {class_dir}")
            for image_path in sorted(p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS):
                info = safe_image_info(image_path)
                records.append(
                    {
                        "sample_id": f"{domain}_{sample_index:07d}",
                        "domain": domain,
                        "label": label,
                        "domain_label": f"{domain}::{label}",
                        "class_folder": folder_name,
                        "file_name": image_path.name,
                        "coco_file_name": normalize_coco_name(f"{folder_name}/{image_path.name}"),
                        "image_path": str(image_path),
                        "relative_path": normalize_coco_name(str(image_path.relative_to(raw_root))),
                        "file_ext": image_path.suffix.lower(),
                        "file_size_bytes": image_path.stat().st_size,
                        "width": info["width"],
                        "height": info["height"],
                        "aspect_ratio": float(info["width"]) / float(info["height"]) if info["is_valid"] else "",
                        "color_mode": info["mode"],
                        "is_valid_image": info["is_valid"],
                    }
                )
                sample_index += 1
    return records


def quality_filter_records(records: list[dict[str, Any]], config: PreprocessConfig) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    seen_md5: set[str] = set()
    iterator = records
    if tqdm is not None:
        iterator = tqdm(records, desc="quality scan", unit="img")
    for record in iterator:
        if not record["is_valid_image"]:
            record["md5"] = ""
            record["blur_score"] = ""
            record["is_duplicate"] = False
            record["is_blurry"] = False
            continue
        path = Path(record["image_path"])
        record["md5"] = md5_file(path)
        record["blur_score"] = blur_score(path) if config.drop_blurry else ""
        record["is_duplicate"] = record["md5"] in seen_md5
        seen_md5.add(record["md5"])

    valid_blurs = [float(r["blur_score"]) for r in records if r["is_valid_image"] and r["blur_score"] != ""]
    global_threshold = percentile(valid_blurs, config.blur_quantile) if config.drop_blurry else float("-inf")
    by_group: dict[str, list[float]] = defaultdict(list)
    for record in records:
        if config.drop_blurry and record["is_valid_image"] and record["blur_score"] != "":
            by_group[record["domain_label"]].append(float(record["blur_score"]))
    thresholds = {
        group: percentile(values, config.blur_quantile) if len(values) >= 40 else global_threshold
        for group, values in by_group.items()
    }

    kept: list[dict[str, Any]] = []
    removed: list[dict[str, Any]] = []
    for record in records:
        threshold = thresholds.get(record["domain_label"], global_threshold)
        record["blur_threshold"] = threshold
        record["is_blurry"] = bool(config.drop_blurry and record["is_valid_image"] and record["blur_score"] != "" and float(record["blur_score"]) < threshold)
        if not record["is_valid_image"]:
            record["removed_reason"] = "invalid"
            removed.append(record)
        elif record["is_duplicate"]:
            record["removed_reason"] = "duplicate"
            removed.append(record)
        elif record["is_blurry"]:
            record["removed_reason"] = "blurry"
            removed.append(record)
        else:
            kept.append(record)
    return kept, removed


def attach_source_annotations(
    records: list[dict[str, Any]], raw_root: Path
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[tuple[str, str], list[dict[str, Any]]]]:
    anns_by_file = {}
    anns_by_file.update(build_source_annotation_index("rice", raw_root / "rice_leaf_disease" / "annotations.coco.json"))
    anns_by_file.update(build_source_annotation_index("coffee", raw_root / "coffee_leaf_disease" / "annotations.coco.json"))

    kept: list[dict[str, Any]] = []
    missing: list[dict[str, Any]] = []
    for record in records:
        key = (record["domain"], record["coco_file_name"])
        source_anns = anns_by_file.get(key, [])
        if not source_anns:
            row = dict(record)
            row["removed_reason"] = "missing_coco_annotation"
            missing.append(row)
            continue
        record["source_annotation_count"] = len(source_anns)
        kept.append(record)
    return kept, missing, anns_by_file


def stratified_split(records: list[dict[str, Any]], config: PreprocessConfig) -> list[dict[str, Any]]:
    if abs(config.train_ratio + config.val_ratio + config.test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0")

    rng = random.Random(config.seed)
    by_group: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        by_group[record["domain_label"]].append(record)

    split_rows: list[dict[str, Any]] = []
    for group in sorted(by_group):
        group_rows = sorted(by_group[group], key=lambda row: row["sample_id"])
        rng.shuffle(group_rows)
        n = len(group_rows)
        n_train = int(round(n * config.train_ratio))
        n_val = int(round(n * config.val_ratio))
        if n_train + n_val > n:
            n_val = max(0, n - n_train)
        assignments = ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val)
        for record, split in zip(group_rows, assignments, strict=True):
            row = dict(record)
            row["split"] = split
            split_rows.append(row)
    return sorted(split_rows, key=lambda row: (row["split"], row["domain"], row["label"], row["sample_id"]))


def resize_with_padding(image: Image.Image, image_size: int) -> tuple[Image.Image, float, int, int]:
    image = image.convert("RGB")
    width, height = image.size
    scale = image_size / max(width, height)
    new_w = max(1, int(round(width * scale)))
    new_h = max(1, int(round(height * scale)))
    resized = image.resize((new_w, new_h), resample=Image.Resampling.BICUBIC)
    pad_x = (image_size - new_w) // 2
    pad_y = (image_size - new_h) // 2
    padding = (pad_x, pad_y, image_size - new_w - pad_x, image_size - new_h - pad_y)
    return ImageOps.expand(resized, border=padding, fill=(0, 0, 0)), scale, pad_x, pad_y


def transform_bbox_xywh(bbox: list[float], scale: float, pad_x: int, pad_y: int, image_size: int) -> list[float]:
    x, y, w, h = [float(v) for v in bbox]
    x1 = min(max(x * scale + pad_x, 0.0), image_size - 1.0)
    y1 = min(max(y * scale + pad_y, 0.0), image_size - 1.0)
    x2 = min(max((x + w) * scale + pad_x, 0.0), float(image_size))
    y2 = min(max((y + h) * scale + pad_y, 0.0), float(image_size))
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]


def transform_polygon(poly: list[float], scale: float, pad_x: int, pad_y: int, image_size: int) -> list[float]:
    out: list[float] = []
    for idx in range(0, len(poly), 2):
        x = min(max(float(poly[idx]) * scale + pad_x, 0.0), image_size - 1.0)
        y = min(max(float(poly[idx + 1]) * scale + pad_y, 0.0), image_size - 1.0)
        out.extend([x, y])
    return out


def bbox_to_polygon_xywh(bbox: list[float]) -> list[list[float]]:
    x, y, w, h = [float(v) for v in bbox]
    if w <= 0 or h <= 0:
        return []
    return [[x, y, x + w, y, x + w, y + h, x, y + h]]


def transform_segmentation(
    segmentation: Any, bbox: list[float], scale: float, pad_x: int, pad_y: int, image_size: int
) -> list[list[float]]:
    if isinstance(segmentation, list) and segmentation:
        transformed = []
        for poly in segmentation:
            if isinstance(poly, list) and len(poly) >= 6 and len(poly) % 2 == 0:
                out_poly = transform_polygon(poly, scale, pad_x, pad_y, image_size)
                if len(out_poly) >= 6:
                    transformed.append(out_poly)
        if transformed:
            return transformed
    return bbox_to_polygon_xywh(transform_bbox_xywh(bbox, scale, pad_x, pad_y, image_size))


def export_images(rows: list[dict[str, Any]], config: PreprocessConfig) -> list[dict[str, Any]]:
    out_rows: list[dict[str, Any]] = []
    image_root = config.output_root / "images"
    iterator = rows
    if tqdm is not None:
        iterator = tqdm(rows, desc="resize/export images", unit="img")
    for row in iterator:
        out_dir = image_root / row["split"] / row["domain"] / label_dirname(row["label"])
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{row['sample_id']}.jpg"
        if config.overwrite or not out_file.exists():
            with Image.open(row["image_path"]) as img:
                resized, scale, pad_x, pad_y = resize_with_padding(img, config.image_size)
                resized.save(out_file, format="JPEG", quality=config.jpeg_quality, optimize=False)
        else:
            with Image.open(row["image_path"]) as img:
                _, scale, pad_x, pad_y = resize_with_padding(img, config.image_size)
        out = dict(row)
        out.update(
            {
                "processed_relative_path": normalize_coco_name(str(out_file.relative_to(config.output_root))),
                "resize_scale": scale,
                "pad_x": pad_x,
                "pad_y": pad_y,
                "processed_width": config.image_size,
                "processed_height": config.image_size,
            }
        )
        out_rows.append(out)
    return out_rows


def export_coco(
    rows: list[dict[str, Any]], anns_by_file: dict[tuple[str, str], list[dict[str, Any]]], config: PreprocessConfig
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    ann_dir = config.output_root / "annotations"
    ann_dir.mkdir(parents=True, exist_ok=True)
    kept_rows: list[dict[str, Any]] = []
    dropped_rows: list[dict[str, Any]] = []
    ann_id = 1
    for split in ["train", "val", "test"]:
        images: list[dict[str, Any]] = []
        annotations: list[dict[str, Any]] = []
        split_rows = [row for row in rows if row["split"] == split]
        for row in split_rows:
            image_id = len(images) + 1
            source_anns = anns_by_file.get((row["domain"], row["coco_file_name"]), [])
            image_annotations = []
            for ann in source_anns:
                bbox = transform_bbox_xywh(
                    ann.get("bbox", [0, 0, 0, 0]), row["resize_scale"], row["pad_x"], row["pad_y"], config.image_size
                )
                if bbox[2] <= 0 or bbox[3] <= 0:
                    continue
                segmentation = transform_segmentation(
                    ann.get("segmentation", []),
                    ann.get("bbox", [0, 0, 0, 0]),
                    row["resize_scale"],
                    row["pad_x"],
                    row["pad_y"],
                    config.image_size,
                )
                if not segmentation:
                    continue
                image_annotations.append(
                    {
                        "id": ann_id,
                        "image_id": image_id,
                        "category_id": CLASS_TO_CATEGORY_ID[row["label"]],
                        "segmentation": segmentation,
                        "bbox": bbox,
                        "area": float(ann.get("area", bbox[2] * bbox[3])) * float(row["resize_scale"]) ** 2,
                        "iscrowd": int(ann.get("iscrowd", 0)),
                    }
                )
                ann_id += 1
            if not image_annotations:
                dropped = dict(row)
                dropped["removed_reason"] = "zero_valid_annotations_after_transform"
                dropped_rows.append(dropped)
                continue
            images.append(
                {
                    "id": image_id,
                    "file_name": row["processed_relative_path"],
                    "width": config.image_size,
                    "height": config.image_size,
                    "domain": row["domain"],
                    "label": row["label"],
                    "sample_id": row["sample_id"],
                }
            )
            annotations.extend(image_annotations)
            kept = dict(row)
            kept["coco_image_id"] = image_id
            kept["annotation_count"] = len(image_annotations)
            kept_rows.append(kept)
        coco = {
            "info": {
                "description": "Mask R-CNN processed rice/coffee leaf disease instance segmentation dataset",
                "source_dataset": "magnusdtd2/rice-coffee-leaf-disease",
                "split": split,
                "preprocessing": f"resize_with_padding_to_{config.image_size}",
            },
            "images": images,
            "annotations": annotations,
            "categories": COCO_CATEGORIES,
        }
        (ann_dir / f"{split}.coco.json").write_text(json.dumps(coco, ensure_ascii=False), encoding="utf-8")
    return kept_rows, dropped_rows


def compute_normalization(rows: list[dict[str, Any]], output_root: Path) -> dict[str, Any]:
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    total_pixels = 0
    train_rows = [row for row in rows if row["split"] == "train"]
    iterator = train_rows
    if tqdm is not None:
        iterator = tqdm(train_rows, desc="compute normalization", unit="img")
    for row in iterator:
        with Image.open(output_root / row["processed_relative_path"]) as img:
            arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
        channel_sum += arr.sum(axis=(0, 1))
        channel_sq_sum += np.square(arr).sum(axis=(0, 1))
        total_pixels += arr.shape[0] * arr.shape[1]
    mean = channel_sum / max(total_pixels, 1)
    var = channel_sq_sum / max(total_pixels, 1) - np.square(mean)
    std = np.sqrt(np.maximum(var, 1e-12))
    return {"source": "mask_rcnn_train_split_only", "num_images": len(train_rows), "mean": mean.tolist(), "std": std.tolist()}


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    keys = sorted({key for row in rows for key in row})
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def count_by(rows: list[dict[str, Any]], keys: tuple[str, ...]) -> list[dict[str, Any]]:
    counts: dict[tuple[Any, ...], int] = defaultdict(int)
    for row in rows:
        counts[tuple(row[key] for key in keys)] += 1
    return [{**{key: value for key, value in zip(keys, group, strict=True)}, "count": count} for group, count in sorted(counts.items())]


def write_metadata(
    config: PreprocessConfig,
    kept_rows: list[dict[str, Any]],
    removed_rows: list[dict[str, Any]],
    missing_rows: list[dict[str, Any]],
    zero_rows: list[dict[str, Any]],
) -> None:
    meta_dir = config.output_root / "metadata"
    meta_dir.mkdir(parents=True, exist_ok=True)
    write_csv(meta_dir / "all_splits_manifest.csv", kept_rows)
    for split in ["train", "val", "test"]:
        write_csv(meta_dir / f"{split}_manifest.csv", [row for row in kept_rows if row["split"] == split])
    write_csv(meta_dir / "removed_quality_rows.csv", removed_rows)
    write_csv(meta_dir / "missing_coco_annotations.csv", missing_rows)
    write_csv(meta_dir / "dropped_zero_annotation_rows.csv", zero_rows)
    write_csv(meta_dir / "class_counts.csv", count_by(kept_rows, ("split", "domain", "label")))
    normalization = compute_normalization(kept_rows, config.output_root)
    (meta_dir / "normalization_stats.json").write_text(json.dumps(normalization, indent=2), encoding="utf-8")
    (meta_dir / "class_names.json").write_text(json.dumps(CLASS_NAMES, indent=2), encoding="utf-8")
    preprocessing_config = {
        **asdict(config),
        "raw_root": str(config.raw_root),
        "output_root": str(config.output_root),
        "class_names": CLASS_NAMES,
        "class_to_index": {name: idx for idx, name in enumerate(CLASS_NAMES)},
        "class_to_category_id": CLASS_TO_CATEGORY_ID,
        "augmentation_policy": "Notebook-level detection transforms must update image, boxes, and masks together.",
        "zero_annotation_policy": "Dropped before writing final manifests and COCO files.",
    }
    (meta_dir / "preprocessing_config.json").write_text(json.dumps(preprocessing_config, indent=2), encoding="utf-8")


def prepare_mask_rcnn_dataset(config: PreprocessConfig) -> dict[str, Any]:
    started = time.time()
    config.output_root.mkdir(parents=True, exist_ok=True)
    print("[1/6] Collecting image records...")
    records = collect_records(config.raw_root)
    print(f"      records={len(records):,}")
    print("[2/6] Quality scan and duplicate detection...")
    quality_kept, quality_removed = quality_filter_records(records, config)
    print(f"      kept={len(quality_kept):,}, removed={len(quality_removed):,}")
    print("[3/6] Matching source COCO annotations...")
    annotation_kept, missing_rows, anns_by_file = attach_source_annotations(quality_kept, config.raw_root)
    print(f"      annotation_kept={len(annotation_kept):,}, missing={len(missing_rows):,}")
    print("[4/6] Stratified split...")
    split_rows = stratified_split(annotation_kept, config)
    print("[5/6] Resizing images and transforming geometry...")
    processed_rows = export_images(split_rows, config)
    final_rows, zero_rows = export_coco(processed_rows, anns_by_file, config)
    print("[6/6] Writing metadata and normalization stats...")
    write_metadata(config, final_rows, quality_removed, missing_rows, zero_rows)
    return {
        "input_records": len(records),
        "quality_kept": len(quality_kept),
        "annotation_kept": len(annotation_kept),
        "final_images": len(final_rows),
        "quality_removed": len(quality_removed),
        "missing_annotations": len(missing_rows),
        "zero_annotation_dropped": len(zero_rows),
        "output_root": str(config.output_root),
        "elapsed_seconds": round(time.time() - started, 2),
    }


In [3]:
def looks_like_raw_root(path: Path) -> bool:
    return (
        path.exists()
        and (path / "rice_leaf_disease" / "annotations.coco.json").exists()
        and (path / "coffee_leaf_disease" / "annotations.coco.json").exists()
    )


def resolve_raw_root(candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        checked.append(candidate)
        if looks_like_raw_root(candidate):
            return candidate
        if candidate.exists() and candidate.is_dir():
            for nested in sorted(candidate.rglob("rice_leaf_disease")):
                root = nested.parent
                checked.append(root)
                if looks_like_raw_root(root):
                    return root
    raise FileNotFoundError(
        "Could not find raw dataset root containing rice_leaf_disease/ and coffee_leaf_disease/.\n"
        + "Checked:\n"
        + "\n".join(str(path) for path in checked[:80])
    )


RAW_ROOT = resolve_raw_root(RAW_ROOT_CANDIDATES)
print("Raw root:", RAW_ROOT)
print("Output root:", OUTPUT_ROOT)

config = PreprocessConfig(
    raw_root=RAW_ROOT,
    output_root=OUTPUT_ROOT,
    image_size=IMAGE_SIZE,
    seed=SEED,
    drop_blurry=DROP_BLURRY,
    blur_quantile=BLUR_QUANTILE,
    overwrite=OVERWRITE,
)
summary = prepare_mask_rcnn_dataset(config)
print(json.dumps(summary, indent=2))


Raw root: /kaggle/input/datasets/magnusdtd2/rice-coffee-leaf-disease
Output root: /kaggle/working/data/mask_rcnn_processed
[1/6] Collecting image records...
      records=7,300
[2/6] Quality scan and duplicate detection...


quality scan:   0%|          | 0/7300 [00:00<?, ?img/s]

      kept=7,299, removed=1
[3/6] Matching source COCO annotations...
      annotation_kept=7,257, missing=42
[4/6] Stratified split...
[5/6] Resizing images and transforming geometry...


resize/export images:   0%|          | 0/7257 [00:00<?, ?img/s]

[6/6] Writing metadata and normalization stats...


compute normalization:   0%|          | 0/5078 [00:00<?, ?img/s]

{
  "input_records": 7300,
  "quality_kept": 7299,
  "annotation_kept": 7257,
  "final_images": 7257,
  "quality_removed": 1,
  "missing_annotations": 42,
  "zero_annotation_dropped": 0,
  "output_root": "/kaggle/working/data/mask_rcnn_processed",
  "elapsed_seconds": 521.98
}


In [4]:
print("Processed dataset is ready:", OUTPUT_ROOT)
print("Metadata files:")
for path in sorted((OUTPUT_ROOT / "metadata").glob("*")):
    print(" -", path, path.stat().st_size, "bytes")
print("Annotation files:")
for path in sorted((OUTPUT_ROOT / "annotations").glob("*.json")):
    coco = json.loads(path.read_text(encoding="utf-8"))
    print(f" - {path.name}: images={len(coco['images'])}, annotations={len(coco['annotations'])}")

if CREATE_ZIP:
    import shutil

    print("Zipping processed dataset. This can take a long time on Kaggle...")
    archive_path = shutil.make_archive(str(ZIP_OUTPUT), "zip", OUTPUT_ROOT)
    print("Created archive:", archive_path)
else:
    print("Zip skipped because CREATE_ZIP=False.")
    print("Use this output directory in training notebooks:", OUTPUT_ROOT)


Processed dataset is ready: /kaggle/working/data/mask_rcnn_processed
Metadata files:
 - /kaggle/working/data/mask_rcnn_processed/metadata/all_splits_manifest.csv 3681603 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/class_counts.csv 636 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/class_names.json 119 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/dropped_zero_annotation_rows.csv 0 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/missing_coco_annotations.csv 17097 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/normalization_stats.json 241 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/preprocessing_config.json 1055 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/removed_quality_rows.csv 724 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/test_manifest.csv 551095 bytes
 - /kaggle/working/data/mask_rcnn_processed/metadata/train_manifest.csv 2582655 bytes
 - /kaggle/working/data/mask_rcnn_proces